# Submission — &lt;Parmesan&gt;

Copy this notebook into `submissions/<team-handle>/submission.ipynb` and
fill in every section. Judges run this notebook top-to-bottom on a clean
machine — make sure it executes.

## 1 · Setup

In [ ]:
import os

import numpy as np
import uniqx

GATEWAY = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
uniqx.login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY)
client = uniqx.connect(GATEWAY)
print("uniqx", uniqx.__version__)

## 2 · Workload

Build your traced module here. Cite the SDK functions you used; comment
any non-obvious choices.

Solving the 2D incompressible Navier-Stokes equation with Chorin's projection method.


$$
\frac{\partial \mathbf{u}}{\partial t} + \left(\mathbf{u} \cdot \nabla\right) \mathbf{u} = - \frac{1}{\rho} \nabla p + \nu \nabla^2 \mathbf{u}
$$

we consider Dirichlet B.C. as specified under `boundary.py` script:

<p align="center">
  <img src="assets/2d_domain.png" alt="2D Domain with Dirichlet Boundary Conditions" width="400"/><br>
  <em>Dirichlet B.C. with top-wall has net inflow $u = U_{\text{lid}}$</em>
</p>



### Methodology: Chorin's Projection Method


<u>Motivation: Helmholtz decomposition</u>

<img src="assets/helmholtz.png" alt="Helmholtz Decomposition Diagram" width="500">

<p align="center"><em>Image Source: <a href="https://www.linkedin.com/posts/alechelbling_the-helmholtz-decomposition-is-one-of-the-ugcPost-7462282011861356544-anpu?utm_source=share&utm_medium=member_desktop&rcm=ACoAADCyUoYByXv17KranNM32Csw2ZBPghg_ZU0">By Alec Helbing</a></em></p>



<u>Derivation of Poisson Equation</u>


In the following, denote "sol - solenoidal",

$$
\mathbf{u} = \mathbf{u}_{\text{sol}} + \mathbf{u}_{\text{irrot}} = \mathbf{u}_{\text{sol}} + \nabla \phi
$$



By taking the divergence on the equation, we arrive at the *Poisson equation* for the scalar function $\phi$

$$
 \begin{align*}
 \nabla \cdot \mathbf{u} &= \nabla \cdot \left(\mathbf{u}_{\text{sol}} + \nabla \phi\right) \\
 &\Leftrightarrow \nabla \cdot \mathbf{u} = \cancel{\nabla \cdot \mathbf{u}_{\text{sol}}} + \nabla \cdot \left(\nabla \phi\right) \\
&\Leftrightarrow \nabla \cdot \mathbf{u} = \nabla^2 \phi
 \end{align*}
$$


We see that for known vector field $\mathbf{u}$, we can solve for the scalar-valued $\phi$; in fact, we consider pressure as the underlying quantity in our application, in which let $\phi := P$ and have the form we see in <u>B - Pressure Poisson</u> with some additional constants $\rho, \Delta t$ for the density, discrete time step size accordingly.



$$
\nabla^2 P = \frac{\rho}{\Delta t} \nabla \cdot \mathbf{u}^{\star}
$$



In [ ]:
# REPLACE — your traced module construction.
# Example pattern:
#   import uniqx
#   @uniqx.to_module(name="my_kernel")
#   def my_kernel(x):
#       return uniqx.ops.matmul(x, x)
#   module = my_kernel
#   runtime_inputs = [your_runtime_data]
module = None
runtime_inputs = []
assert module is not None, "build your traced module above"

## 3 · Preflight — the Pareto table

Required: print `options.summary()` and copy the output into
`preflight_log.txt`.

In [ ]:
options = uniqx.preflight(module, client=client)
print(options.summary())
try:
    options.plot()
except Exception as exc:
    print(f"plot skipped: {exc}")

## 4 · Submit

State which option you picked and *why* in the markdown cell below the
submission. The `why` paragraph goes verbatim into `results.json.justification`.

In [ ]:
choice = options.recommended    # or pick a non-recommended row
print(f"Selected: {choice['label']}")

job_id = uniqx.submit(
    module,
    client=client,
    preflight_job_id=options.job_id,
    option_idx=choice["_idx"],
    runtime_inputs=runtime_inputs,
)
result = uniqx.get(job_id, client=client, timeout=600)
print("job complete")

**Why this option:** REPLACE — 3-6 sentences referencing specific numbers
from the preflight table above.

## 5 · Baseline comparison

In [ ]:
# REPLACE — run your reference (PySCF / NumPy / custom) and compute
# the metrics you'll record in results.json.metrics.
runtime_s = 0.0
accuracy_rel_error = 0.0
print(f"runtime_s          : {runtime_s}")
print(f"accuracy_rel_error : {accuracy_rel_error}")

## 6 · Discussion

REPLACE — short prose answering:

1. What did the Pareto frontier look like?
2. What did you learn from comparing baselines?
3. Where would you push next with more time?